In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Найпоширеніші параметри транспіляції

<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці було розроблено з використанням таких залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ця сторінка описує деякі з найпоширеніших параметрів для локальної транспіляції. Ці параметри налаштовуються через аргументи [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) або [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Ступінь апроксимації
Ступінь апроксимації дозволяє вказати, наскільки точно результуюча схема має відповідати бажаній (вхідній) схемі. Це число з плаваючою точкою в діапазоні (0.0 – 1.0), де 0.0 — максимальна апроксимація, а 1.0 (за замовчуванням) — без апроксимації. Менші значення обмінюють точність виводу на простоту виконання (тобто менше вентилів). Значення за замовчуванням — 1.0.

У синтезі двокубітних унітарних операцій (який використовується на початкових етапах для всіх рівнів та на етапі оптимізації з рівнем 3) це значення задає цільову точність (fidelity) вихідного розкладання. Тобто, скільки похибки вноситься при перетворенні матричного представлення схеми на дискретні вентилі. Якщо ступінь апроксимації менший (більша апроксимація), вихідна схема після синтезу відрізнятиметься більше від вхідної матриці, але, ймовірно, матиме менше вентилів (оскільки будь-яку довільну двокубітну операцію можна розкласти точно щонайбільше трьома вентилями CX) і її легше виконати.

Коли ступінь апроксимації менший за 1.0, може бути синтезовано схеми з одним або двома вентилями CX, що призводить до меншої апаратної похибки, але більшої похибки апроксимації. Оскільки CX є найдорожчим вентилем з точки зору похибки, може бути вигідно зменшити їхню кількість ціною точності синтезу (ця техніка використовувалась для збільшення квантового об'єму на пристроях IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Як приклад, ми генеруємо випадковий двокубітний `UnitaryGate`, який буде синтезовано на початковому етапі. Встановлення `approximation_degree` меншим за 1.0 може сформувати наближену схему. Також треба вказати `basis_gates`, щоб метод синтезу знав, які вентилі він може використовувати для наближеного синтезу.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Результат дорівнює `2`, оскільки апроксимація потребує менше вентилів CX.

<span id="seed"></span>
## Зерно генератора випадкових чисел
Деякі частини Transpiler є стохастичними, тому повторні запуски транспіляції можуть повертати різні результати. Щоб отримати відтворюваний результат, можна встановити зерно для генератора псевдовипадкових чисел за допомогою аргументу `seed_transpiler`. Повторні запуски з однаковим зерном повертатимуть однакові результати.

Приклад:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Початкове розміщення
До транспіляції кубіти у твоїй схемі є віртуальними і не обов'язково відповідають фізичним кубітам цільового Backend. Можна задати початкове відображення віртуальних кубітів на фізичні за допомогою аргументу `initial_layout`. Зверни увагу, що кінцеве розміщення кубітів може відрізнятися від початкового, оскільки Transpiler може переставляти кубіти за допомогою swap-вентилів або інших засобів.

У прикладі нижче ми будуємо початкове розміщення для мок-Backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke), створюючи об'єкт [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Наше розміщення відображає перший кубіт схеми на кубіт 5 Sherbrooke, а другий кубіт схеми — на кубіт 6 Sherbrooke. Зверни увагу, що фізичні кубіти завжди представлені цілими числами.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Крім вказання об'єкта Layout, можна також передати список цілих чисел, де $i$-й елемент списку містить фізичний кубіт, на який слід відобразити $i$-й кубіт. Наприклад:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Вивід попередньої комірки коду](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Можна скористатися функцією [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map), щоб сформувати діаграму графа пристрою з інформацією про похибки та підписаними фізичними кубітами. Аналогічні діаграми також доступні на сторінці [Обчислювальні ресурси](https://quantum.cloud.ibm.com/computers).